# E-Commerce Recommendation System

End-to-end demo using **Ray Data → Ray Train → Ray Serve** on Anyscale.

```
┌──────────────────────────────────────────────────────────────────────┐
│  Product catalog (synthetic)                                         │
│       │                                                              │
│       ▼  Stage 1 — Ray Data                                         │
│  Preprocess images (resize, normalise)                               │
│  Clean text (descriptions + category)                               │
│       │                                                              │
│       ▼  Stage 2 — Ray Train (TorchTrainer)                         │
│  Fine-tune all-MiniLM-L6-v2 with contrastive loss                  │
│  Save model + pre-compute product embeddings                        │
│       │                                                              │
│       ▼  Stage 3 — Ray Serve                                        │
│  POST /recommend  {image_base64}                                     │
│    ImageToText  (BLIP-base)  →  caption                             │
│    ProductRecommender  (MiniLM)  →  top-5 products                  │
│       │                                                              │
│       ▼  Streamlit UI                                               │
│  Upload image → see recommendations                                 │
└──────────────────────────────────────────────────────────────────────┘
```

**Models used**
| Stage | Model | Size | Device |
|-------|-------|------|--------|
| Train | `sentence-transformers/all-MiniLM-L6-v2` | 22 M params | CPU |
| Serve | `Salesforce/blip-image-captioning-base` | 224 M params | CPU |
| Serve | fine-tuned MiniLM | 22 M params | CPU |

Everything runs on **CPU** and completes in a few minutes.

## 0 — Setup

In [ ]:
# Install dependencies (skip if already installed)
# !pip install -q -r requirements.txt

In [ ]:
import json
import os
import sys
import numpy as np
from pathlib import Path
from PIL import Image

# Make sure the project root is on the path
sys.path.insert(0, str(Path().resolve()))

from utils import PRODUCTS, CATEGORIES, make_product_image

print(f"Products in catalog : {len(PRODUCTS)}")
print(f"Categories          : {CATEGORIES}")

### Peek at the synthetic product images

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, cat in zip(axes, CATEGORIES):
    product = next(p for p in PRODUCTS if p["category"] == cat)
    img = make_product_image(product, seed=0)
    ax.imshow(img)
    ax.set_title(cat, fontsize=9)
    ax.axis("off")
plt.suptitle("One sample image per category", fontsize=12)
plt.tight_layout()
plt.show()

---
## Stage 1 — Ray Data: Preprocessing

In [ ]:
# Run the preprocessing pipeline
# (generates data/raw/ and data/preprocessed/)
%run preprocess_pipeline.py

In [ ]:
import ray
ray.init(ignore_reinit_error=True)

ds = ray.data.read_parquet("data/preprocessed")
print(f"Rows: {ds.count()}")
print(f"Schema:\n{ds.schema()}")

sample = ds.take(2)
for row in sample:
    print(f"  [{row['product_id']}] {row['name']!r}  "
          f"img_bytes={len(row['image_tensor_bytes'])}  "
          f"text={row['text_clean'][:60]}…")

---
## Stage 2 — Ray Train: Embedding Fine-Tuning

In [ ]:
# Fine-tune all-MiniLM-L6-v2 and export embeddings
# (writes models/embedding_model/, models/product_embeddings.npy,
#  models/product_metadata.json)
%run train_embedding.py

In [ ]:
# Inspect the computed embeddings
embeddings = np.load("models/product_embeddings.npy")
with open("models/product_metadata.json") as f:
    metadata = json.load(f)

print(f"Embeddings shape : {embeddings.shape}")
print(f"Products indexed : {len(metadata)}")

# Cosine similarity heatmap between first 10 products
n = min(10, len(embeddings))
sub = embeddings[:n]
norms = np.linalg.norm(sub, axis=1, keepdims=True)
sim = (sub / norms) @ (sub / norms).T

labels = [m["name"][:20] for m in metadata[:n]]
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim, vmin=-1, vmax=1, cmap="RdYlGn")
ax.set_xticks(range(n)); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(n)); ax.set_yticklabels(labels, fontsize=8)
plt.colorbar(im, ax=ax, label="Cosine similarity")
ax.set_title("Product embedding similarity (top 10)")
plt.tight_layout()
plt.show()

---
## Stage 3 — Ray Serve: Online Recommendation API

In [ ]:
import ray
from ray import serve

ray.init(ignore_reinit_error=True)

# Deploy locally
from serve_app import app
serve.run(app, host="0.0.0.0", port=8000)
print("Service running at http://localhost:8000")

In [ ]:
# --- Test the endpoint ---
import base64, io, requests
from utils import make_product_image, PRODUCTS

# Build a test image (Wireless Headphones)
product = PRODUCTS[0]
img_arr = make_product_image(product, seed=0)
img_pil = Image.fromarray(img_arr)

buf = io.BytesIO()
img_pil.save(buf, format="JPEG")
encoded = base64.b64encode(buf.getvalue()).decode()

resp = requests.post("http://localhost:8000/recommend", json={"image_base64": encoded})
result = resp.json()

print(f"Caption : {result['caption']!r}")
print(f"\nTop-{len(result['recommendations'])} recommendations:")
for r in result["recommendations"]:
    print(f"  {r['rank']}. [{r['category']:18s}] {r['name']:35s}  sim={r['similarity']:.3f}")

In [ ]:
# Show the query image alongside recommendations
fig, axes = plt.subplots(1, 6, figsize=(18, 3))

axes[0].imshow(img_arr)
axes[0].set_title("Query image", fontsize=9, color="navy", fontweight="bold")
axes[0].axis("off")

with open("models/product_metadata.json") as f:
    meta = {m["product_id"]: m for m in json.load(f)}

all_prods = {p["name"]: p for p in PRODUCTS}

for ax, rec in zip(axes[1:], result["recommendations"]):
    prod_name = rec["name"]
    prod = all_prods.get(prod_name, {"name": prod_name, "category": rec["category"]})
    rec_img = make_product_image({**prod, "category": rec["category"]}, seed=42)
    ax.imshow(rec_img)
    ax.set_title(
        f"{rec['rank']}. {rec['name'][:18]}\n{rec['similarity']:.2f}",
        fontsize=8
    )
    ax.axis("off")

plt.suptitle(f'Caption: "{result["caption"]}"', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Teardown (optional — keeps the service running if you want to use the Streamlit app)
# serve.shutdown()

---
## Streamlit UI

With the service running, open a terminal and run:

```bash
streamlit run streamlit_app.py
```

Then navigate to **http://localhost:8501** in your browser.

---

## Deploy to Anyscale

```bash
# Stage 1 — preprocess
anyscale job submit -f job_preprocess.yaml

# Stage 2 — train
anyscale job submit -f job_train.yaml

# Stage 3 — serve
anyscale service deploy -f service.yaml
```